In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


spark = SparkSession.builder.getOrCreate()

BASE_PATH = "/home/jovyan/work/output"

orders = spark.read.csv(f"{BASE_PATH}/bronze/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv(f"{BASE_PATH}/bronze/order_items.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{BASE_PATH}/bronze/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{BASE_PATH}/bronze/products.csv", header=True, inferSchema=True)
sellers = spark.read.csv(f"{BASE_PATH}/bronze/sellers.csv", header=True, inferSchema=True)
payments = spark.read.csv(f"{BASE_PATH}/bronze/payments.csv", header=True, inferSchema=True)

In [15]:
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)



In [13]:
products_silver.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)
 |-- _is_valid: boolean (nullable = false)



In [18]:
dim_products = products_silver \
    .withColumn(
        "product_category_name",
        coalesce(col("product_category_name"), lit("unknown"))
    ) \
    .withColumn(
        "product_volume_cm3",
        when(
            col("product_length_cm").isNotNull() &
            col("product_height_cm").isNotNull() &
            col("product_width_cm").isNotNull(),
            col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
        )
    ) \
    .withColumn("product_key", abs(hash(col("product_id")))) \
    .select(
        "product_key",
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_volume_cm3",
        "product_photos_qty",
        "product_description_length"
    )

In [21]:
dim_sellers = sellers_silver \
    .withColumn("seller_key", abs(hash(col("seller_id")))) \
    .select(
        "seller_key",
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code_prefix"
    )

In [22]:
cust_orders = customers_silver.join(orders_silver, "customer_id")

In [11]:
cust_orders.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)



In [23]:
from pyspark.sql.functions import to_timestamp

cust_orders = cust_orders.withColumn(
    "order_purchase_timestamp",
    to_timestamp("order_purchase_timestamp")
)
window_latest = Window.partitionBy("customer_unique_id").orderBy(col("order_purchase_timestamp").desc())

latest_customer = cust_orders \
    .withColumn("rn", row_number().over(window_latest)) \
    .filter(col("rn") == 1)

In [24]:
customer_agg = cust_orders \
    .join(order_items_silver, "order_id") \
    .groupBy("customer_unique_id") \
    .agg(
        min("order_purchase_timestamp").alias("first_order_date"),
        countDistinct("order_id").alias("total_orders"),
        sum(col("price") + col("freight_value")).alias("total_spend")
    )

In [28]:
order_total = order_items_silver.groupBy("order_id").agg(
    sum(col("price") + col("freight_value")).alias("order_total")
)

cust_spend = cust_orders.join(order_total, "order_id") \
    .groupBy("customer_unique_id") \
    .agg(sum("order_total").alias("total_spend"))

In [29]:
dim_customers = latest_customer \
    .join(customer_agg, "customer_unique_id") \
    .withColumn("customer_key", abs(hash(col("customer_unique_id")))) \
    .withColumn("is_repeat_customer", col("total_orders") > 1) \
    .select(
        "customer_key",
        "customer_unique_id",
        col("customer_city"),
        col("customer_state"),
        col("customer_zip_code_prefix"),
        "first_order_date",
        "total_orders",
        "total_spend",
        "is_repeat_customer"
    )

In [38]:
fact = order_items_silver \
    .join(orders_silver, "order_id") \
    .join(customers_silver.select("customer_id", "customer_unique_id"), "customer_id") \
    .join(dim_customers, "customer_unique_id") \
    .join(dim_products, "product_id") \
    .join(dim_sellers, "seller_id") \
    .join(order_payment_total, "order_id", "left") \
    .join(payment_type_df, "order_id", "left") \
    .join(payment_installments_df, "order_id", "left")

In [32]:
order_payment_total = payments_silver.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_payment"))

In [33]:
payment_type_window = Window.partitionBy("order_id").orderBy(desc("payment_value"), asc("payment_type"))

payment_type_df = payments_silver \
    .withColumn("rn", row_number().over(payment_type_window)) \
    .filter("rn = 1") \
    .select("order_id", col("payment_type"))

In [40]:
order_total_price = order_items_silver.groupBy("order_id") \
    .agg(sum("price").alias("order_total_price"))

In [37]:
payment_installments_df = payments_silver.groupBy("order_id") \
    .agg(max("payment_installments").alias("payment_installments"))

In [41]:
fact = fact.join(order_total_price, "order_id")

fact = fact.withColumn(
    "payment_value",
    col("total_payment") * (col("price") / col("order_total_price"))
)

In [24]:
fact.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)
 |-- customer_key: string (nullable = true)
 |-- product_key:

In [42]:
fact = fact \
    .withColumn(
        "days_to_deliver",
        when(col("order_delivered_customer_date").isNotNull(),
             datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    ) \
    .withColumn(
        "days_delivery_vs_estimate",
        when(col("order_delivered_customer_date").isNotNull(),
             datediff("order_delivered_customer_date", "order_estimated_delivery_date"))
    ) \
    .withColumn(
        "is_late_delivery",
        when(col("days_delivery_vs_estimate") > 0, True)
    )

In [43]:
fact_order_items = fact \
    .withColumn("order_item_sk", abs(hash(col("order_id"), col("order_item_id")))) \
    .select(
        "order_item_sk",
        "order_id",
        "order_item_id",
        "customer_key",
        "product_key",
        "seller_key",
        to_date("order_purchase_timestamp").alias("order_date"),
        "order_status",
        "price",
        "freight_value",
        "payment_value",
        "payment_type",
        "payment_installments",
        "days_to_deliver",
        "days_delivery_vs_estimate",
        "is_late_delivery"
    )

In [44]:
fact_order_items.write.mode("overwrite").csv(f"{BASE_PATH}/gold/fact_order_items", header=True)
dim_customers.write.mode("overwrite").csv(f"{BASE_PATH}/gold/dim_customers", header=True)
dim_products.write.mode("overwrite").csv(f"{BASE_PATH}/gold/dim_products", header=True)
dim_sellers.write.mode("overwrite").csv(f"{BASE_PATH}/gold/dim_sellers", header=True)

In [45]:
print("Fact:", fact.count())
print("Customers:", dim_customers.count())
print("Products:", dim_products.count())
print("Sellers:", dim_sellers.count())

Fact: 112650
Customers: 95420
Products: 32951
Sellers: 3095


In [32]:
BASE_PATH = "/home/jovyan/work/output"

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

orders_silver = orders \
    .dropDuplicates() \
    .withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast(TimestampType())) \
    .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast(TimestampType())) \
    .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast(TimestampType())) \
    .withColumn(
        "_is_valid",
        when(
            col("order_delivered_customer_date") < col("order_purchase_timestamp"),
            False
        ).otherwise(True)
    )

In [3]:
order_items_silver = order_items \
    .dropDuplicates() \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn(
        "_is_valid",
        when(col("price") < 0, False).otherwise(True)
    )

In [4]:
order_items_silver = order_items \
    .dropDuplicates() \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn(
        "_is_valid",
        when(col("price") < 0, False).otherwise(True)
    )

In [5]:
customers_silver = customers \
    .dropDuplicates() \
    .withColumn("_is_valid", lit(True))

In [6]:
products_silver = products \
    .dropDuplicates() \
    .withColumn("_is_valid", lit(True))

In [17]:
products_silver = products_silver \
    .withColumnRenamed("product_description_lenght", "product_description_length") \
    .withColumnRenamed("product_name_lenght", "product_name_length")

In [7]:
sellers_silver = sellers \
    .dropDuplicates() \
    .withColumn("_is_valid", lit(True))

In [8]:
payments_silver = payments \
    .dropDuplicates() \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("_is_valid", lit(True))

In [9]:
valid_orders = orders_silver.select("order_id")

order_items_silver = order_items_silver.join(
    valid_orders,
    "order_id",
    "left"
).withColumn(
    "_is_valid",
    when(col("order_id").isNull(), False).otherwise(col("_is_valid"))
)

In [10]:
valid_orders = orders_silver.select("order_id")

order_items_silver = order_items_silver.join(
    valid_orders,
    "order_id",
    "left"
).withColumn(
    "_is_valid",
    when(col("order_id").isNull(), False).otherwise(col("_is_valid"))
)

In [46]:
print("Fact:", fact.count())
print("Customers:", dim_customers.count())
print("Products:", dim_products.count())
print("Sellers:", dim_sellers.count())

Fact: 112650
Customers: 95420
Products: 32951
Sellers: 3095


In [47]:
fact.join(dim_customers, "customer_key", "left_anti").show()
fact.join(dim_products, "product_key", "left_anti").show()
fact.join(dim_sellers, "seller_key", "left_anti").show()

+------------+--------+---------+----------+------------------+-----------+-------------+-------------------+-----+-------------+------------+----------------+---------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+----------------+---------+-------------+--------------+------------------------+----------------+------------+-----------+------------------+-----------+---------------------+----------------+------------------+------------------+--------------------------+----------+-----------+------------+----------------------+-------------+------------+--------------------+-----------------+-------------+---------------+-------------------------+----------------+
|customer_key|order_id|seller_id|product_id|customer_unique_id|customer_id|order_item_id|shipping_limit_date|price|freight_value|_ingested_at|_source_endpoint|_is_valid|order_status|order_purchase_timestamp|order_app

In [20]:
orders_silver.write.mode("overwrite").csv(f"{BASE_PATH}/silver/orders", header=True)
order_items_silver.write.mode("overwrite").csv(f"{BASE_PATH}/silver/order_items", header=True)
customers_silver.write.mode("overwrite").csv(f"{BASE_PATH}/silver/customers", header=True)
products_silver.write.mode("overwrite").csv(f"{BASE_PATH}/silver/products", header=True)
sellers_silver.write.mode("overwrite").csv(f"{BASE_PATH}/silver/sellers", header=True)
payments_silver.write.mode("overwrite").csv(f"{BASE_PATH}/silver/payments", header=True)

In [48]:
fact_order_items.createOrReplaceTempView("fact_order_items")
dim_products.createOrReplaceTempView("dim_products")